# MedNeXt MEN-RT Smoke Test (50 cases, 5 folds, 30 epochs)
**Dataset required:** BraTS-MEN-RT (attach as input dataset with path `/kaggle/input/brats-men-rt/`)

**Before running:** Enable Internet in notebook settings (right panel > Internet > ON)

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Clone the code repository
import os
os.chdir('/kaggle/working')
!git clone https://github.com/maryampervaiz-alt/brats-menrt-mednext-thesis.git
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
!pwd && ls

In [ ]:
# Step 3: Install dependencies
!pip install -q pyyaml simpleitk scipy pandas matplotlib tensorboard
!pip install -q nnunet==1.7.1
!pip install -q git+https://github.com/MIC-DKFZ/MedNeXt.git

In [ ]:
# Step 4: Verify dataset is accessible
import os
train_root = '/kaggle/input/brats-men-rt/BraTS2024-MEN-RT-TrainingData'
val_root = '/kaggle/input/brats-men-rt/BraTS2024-MEN-RT-ValidationData'
print('Train cases:', len(os.listdir(train_root)) if os.path.exists(train_root) else 'NOT FOUND')
print('Val cases:', len(os.listdir(val_root)) if os.path.exists(val_root) else 'NOT FOUND')

In [ ]:
# Step 5: Validate setup
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
!python scripts/validate_mednext_nnunet_setup.py --config configs/mednext_nnunet.yaml --check-trainer

In [ ]:
# Step 6: Prepare dataset (copies 50 train cases into nnUNet format)
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
!python scripts/run_mednext_nnunet.py --config configs/mednext_nnunet.yaml --mode prepare

In [ ]:
# Step 7: Plan and preprocess (nnUNet auto-configures patch/batch size)
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
!python scripts/run_mednext_nnunet.py --config configs/mednext_nnunet.yaml --mode preprocess

In [ ]:
# Step 8: Create stratified cross-validation splits
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
!python scripts/run_mednext_nnunet.py --config configs/mednext_nnunet.yaml --mode make-splits

In [ ]:
# Step 9: Install custom MedNeXt trainer
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
!python scripts/run_mednext_nnunet.py --config configs/mednext_nnunet.yaml --mode install-trainer

In [ ]:
# Step 10: Train all 5 folds (30 epochs each)
os.chdir('/kaggle/working/brats-menrt-mednext-thesis')
for fold in range(5):
    print(f'\n========== FOLD {fold} ==========')
    os.system(f'python scripts/run_mednext_nnunet.py --config configs/mednext_nnunet.yaml --mode train --fold {fold} --max-epochs 30')

In [ ]:
# Step 11: Save results to output (Kaggle saves /kaggle/working automatically)
!ls /kaggle/working/nnUNet_results/